# 📖 Notebook 1: DNS and Load Balancing

📖 **Source**: [Hello Interview – Networking Essentials](https://www.hellointerview.com/learn/system-design/core-concepts/networking-essentials)

When you type `google.com` into your browser, how does your computer know which server to talk to? And when Google has *thousands* of servers, how does your request end up at the right one? The answer is **DNS** (for finding servers) and **Load Balancing** (for choosing which one).

## Learning Objectives

By the end of this notebook, you'll understand:
- How DNS translates domain names to IP addresses
- How DNS itself acts as a simple form of load balancing
- What a reverse proxy / load balancer does (using nginx)
- The difference between round-robin, least-connections, IP-hash, and weighted algorithms
- How health checks keep your system reliable

## 🛠️ Setup

Start the infrastructure first:

```bash
cd core-concepts/networking-essentials
docker-compose up -d --build
```

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).  
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [1]:
import socket
import requests
import time
from collections import Counter

# Base URL for our nginx load balancer running in Docker
NGINX_URL = "http://localhost:8080"

# Quick check that the infrastructure is running
try:
    r = requests.get(f"{NGINX_URL}/nginx-health", timeout=3)
    print(f"✅ Nginx is up: {r.text.strip()}")
except requests.ConnectionError:
    print("❌ Nginx is not running. Start it with: docker-compose up -d --build")

✅ Nginx is up: nginx is healthy


---
## Part 1: How DNS Works

**DNS** (Domain Name System) is like the phone book of the internet. Humans remember names (`google.com`), but computers communicate using IP addresses (`142.250.80.46`).

### The DNS Hierarchy

DNS is a **distributed, hierarchical** system:

```
You type google.com
    ↓
Your computer checks its local cache
    ↓ (cache miss)
Asks your ISP's DNS resolver
    ↓ (cache miss)
Asks a Root DNS server → "Try the .com servers"
    ↓
Asks the .com TLD server → "google.com is managed by ns1.google.com"
    ↓
Asks Google's authoritative DNS → "142.250.80.46"
    ↓
Your browser connects to 142.250.80.46
```

Let's see this in action!

In [2]:
# === DNS Lookup with Python ===
# socket.getaddrinfo() does the same thing your browser does:
# it asks the OS to resolve a domain name to IP addresses.

domains = ["google.com", "github.com", "example.com"]

for domain in domains:
    try:
        # AF_INET = IPv4, SOCK_STREAM = TCP
        results = socket.getaddrinfo(domain, 80, socket.AF_INET, socket.SOCK_STREAM)
        # Each result is (family, type, proto, canonname, sockaddr)
        ips = sorted({r[4][0] for r in results})
        print(f"{domain:20s} -> {ips}")
    except socket.gaierror as e:
        # Happens if you're offline or the resolver is having a bad moment.
        print(f"{domain:20s} -> DNS lookup failed ({e}). Are you online?")


google.com           -> ['142.250.75.206']
github.com           -> ['20.217.135.5']
example.com          -> DNS lookup failed ([Errno 8] nodename nor servname provided, or not known). Are you online?


In [3]:
# === DNS as Load Balancing ===
# Large sites return MULTIPLE IP addresses.
# Your client picks one -- effectively distributing load across servers!
#
# Run this cell several times -- you may see the order of IPs change.
# This rotation is called "DNS round-robin".

domain = "google.com"
print(f"Looking up {domain} three times:\n")

for i in range(3):
    try:
        results = socket.getaddrinfo(domain, 443, socket.AF_INET, socket.SOCK_STREAM)
        ips = [r[4][0] for r in results]
        print(f"  Attempt {i+1}: {ips}")
    except socket.gaierror as e:
        print(f"  Attempt {i+1}: lookup failed ({e})")
    time.sleep(0.5)

print("\nDNS often returns multiple IPs so clients spread across different servers.")
print("This is 'client-side load balancing' -- the simplest form of LB!")


Looking up google.com three times:

  Attempt 1: ['142.250.75.206']


  Attempt 2: ['142.250.75.206']


  Attempt 3: ['142.250.75.206']



DNS often returns multiple IPs so clients spread across different servers.
This is 'client-side load balancing' -- the simplest form of LB!


### DNS TTL (Time To Live)

DNS records have a **TTL** — how long the answer can be cached. This matters because:
- **Short TTL** (e.g., 60 seconds) → changes propagate fast, but more DNS lookups
- **Long TTL** (e.g., 24 hours) → fewer lookups, but changes take time to reach everyone

In system design, if you need to quickly redirect traffic (e.g., during a failover), you want a **short TTL**.

---
## Part 2: Load Balancing with Nginx

DNS load balancing is simple but limited — it can't check if a server is healthy, and clients cache results. For real traffic management, we use a **dedicated load balancer**.

Our Docker setup has:
- **3 Flask backends** (`backend1`, `backend2`, `backend3`) — identical servers
- **1 Nginx** — sits in front and decides which backend gets each request

```
Client → Nginx (load balancer) → backend1
                                → backend2
                                → backend3
```

This is a **Layer 7 (L7) load balancer** because nginx understands HTTP. It can inspect URLs, headers, and cookies to make routing decisions.

### Algorithm 1: Round-Robin (Default)

The simplest algorithm: requests go to servers **in order**, cycling through them.

```
Request 1 → backend1
Request 2 → backend2
Request 3 → backend3
Request 4 → backend1  (back to the start)
```

In [4]:
# === Round-Robin Load Balancing ===
# Send 12 requests to nginx and see which backend handles each one.

print("Round-Robin — sending 12 requests to nginx:\n")

servers_hit = []
for i in range(12):
    r = requests.get(NGINX_URL)
    data = r.json()
    server = data["server"]
    servers_hit.append(server)
    print(f"  Request {i+1:2d} → {server}")

print(f"\n📊 Distribution: {dict(Counter(servers_hit))}")
print("💡 Notice how requests cycle through backends evenly!")

Round-Robin — sending 12 requests to nginx:

  Request  1 → backend2
  Request  2 → backend3
  Request  3 → backend1
  Request  4 → backend2
  Request  5 → backend3
  Request  6 → backend1
  Request  7 → backend2
  Request  8 → backend3
  Request  9 → backend1
  Request 10 → backend2
  Request 11 → backend3
  Request 12 → backend1

📊 Distribution: {'backend2': 4, 'backend3': 4, 'backend1': 4}
💡 Notice how requests cycle through backends evenly!


### Algorithm 2: Least Connections

Send each request to the server with the **fewest active connections**. This is smarter than round-robin when some requests take longer than others.

```
backend1: 5 active connections
backend2: 2 active connections  ← next request goes here
backend3: 4 active connections
```

In [5]:
# === Least Connections ===
# We send many requests to the /least-conn/ path which uses least_conn.

import concurrent.futures

def make_request(url):
    """Make a request and return which server handled it."""
    r = requests.get(url)
    return r.json()["server"]

print("Least Connections — sending 15 requests:\n")

servers_hit = []
for i in range(15):
    server = make_request(f"{NGINX_URL}/least-conn/")
    servers_hit.append(server)
    print(f"  Request {i+1:2d} → {server}")

print(f"\n📊 Distribution: {dict(Counter(servers_hit))}")
print("💡 With quick sequential requests, this looks similar to round-robin.")
print("   The difference shows up when some requests are SLOW (see next cell).")

Least Connections — sending 15 requests:



  Request  1 → backend1
  Request  2 → backend2


  Request  3 → backend3
  Request  4 → backend1
  Request  5 → backend2
  Request  6 → backend3
  Request  7 → backend1
  Request  8 → backend2
  Request  9 → backend3
  Request 10 → backend1
  Request 11 → backend2
  Request 12 → backend3
  Request 13 → backend1
  Request 14 → backend2
  Request 15 → backend3

📊 Distribution: {'backend1': 5, 'backend2': 5, 'backend3': 5}
💡 With quick sequential requests, this looks similar to round-robin.
   The difference shows up when some requests are SLOW (see next cell).


In [6]:
# === Least Connections with Slow Requests ===
# First, start some slow requests in the background.
# Then send fast requests — they should avoid the busy server.

print("Simulating: 3 slow requests (2s each) + 9 fast requests in parallel\n")

results = []

with concurrent.futures.ThreadPoolExecutor(max_workers=12) as pool:
    # Start 3 slow requests (these will occupy backends)
    slow_futures = [
        pool.submit(make_request, f"{NGINX_URL}/least-conn/slow?delay=2")
        for _ in range(3)
    ]
    time.sleep(0.3)  # let slow requests connect

    # Now send fast requests — least_conn should route around busy servers
    fast_futures = [
        pool.submit(make_request, f"{NGINX_URL}/least-conn/")
        for _ in range(9)
    ]

    slow_results = [f.result() for f in slow_futures]
    fast_results = [f.result() for f in fast_futures]

print(f"Slow requests handled by: {slow_results}")
print(f"Fast requests handled by: {fast_results}")
print(f"\n📊 Fast request distribution: {dict(Counter(fast_results))}")
print("💡 Least-connections avoids servers that are busy with slow requests!")

Simulating: 3 slow requests (2s each) + 9 fast requests in parallel



Slow requests handled by: ['backend2', 'backend1', 'backend3']
Fast requests handled by: ['backend3', 'backend2', 'backend1', 'backend3', 'backend2', 'backend1', 'backend3', 'backend1', 'backend3']

📊 Fast request distribution: {'backend3': 4, 'backend2': 2, 'backend1': 3}
💡 Least-connections avoids servers that are busy with slow requests!


### Algorithm 3: IP Hash (Sticky Sessions)

The same client IP **always** goes to the same backend. This is useful when backends store session state in memory.

```
Client 10.0.0.1 → always backend2
Client 10.0.0.2 → always backend1
Client 10.0.0.3 → always backend3
```

In [7]:
# === IP Hash (Sticky Sessions) ===
# Every request from our machine should go to the SAME backend.

print("IP Hash — sending 10 requests (all from the same IP):\n")

servers_hit = []
for i in range(10):
    r = requests.get(f"{NGINX_URL}/ip-hash/")
    server = r.json()["server"]
    servers_hit.append(server)
    print(f"  Request {i+1:2d} → {server}")

print(f"\n📊 Distribution: {dict(Counter(servers_hit))}")
print("💡 All requests go to the same backend — that's 'sticky sessions'!")
print("   Useful when servers store session data in memory.")

IP Hash — sending 10 requests (all from the same IP):

  Request  1 → backend3
  Request  2 → backend3
  Request  3 → backend3
  Request  4 → backend3
  Request  5 → backend3
  Request  6 → backend3
  Request  7 → backend3
  Request  8 → backend3
  Request  9 → backend3
  Request 10 → backend3

📊 Distribution: {'backend3': 10}
💡 All requests go to the same backend — that's 'sticky sessions'!
   Useful when servers store session data in memory.


### Algorithm 4: Weighted Round-Robin

Give more traffic to more powerful servers. In our config, `backend1` has weight 3, while `backend2` and `backend3` have weight 1. So `backend1` gets 3× the traffic.

In [8]:
# === Weighted Round-Robin ===
# backend1 has weight=3, backend2 and backend3 have weight=1.
# So out of every 5 requests, backend1 should get 3.

print("Weighted Round-Robin — sending 20 requests:\n")

servers_hit = []
for i in range(20):
    r = requests.get(f"{NGINX_URL}/weighted/")
    server = r.json()["server"]
    servers_hit.append(server)

counts = Counter(servers_hit)
total = sum(counts.values())

print("📊 Distribution:")
for server in sorted(counts):
    pct = counts[server] / total * 100
    bar = "█" * int(pct / 2)
    print(f"  {server}: {counts[server]:2d} requests ({pct:.0f}%) {bar}")

print("\n💡 backend1 gets ~60% of traffic (weight 3/5).")
print("   Use this when some servers are more powerful than others.")

Weighted Round-Robin — sending 20 requests:



📊 Distribution:
  backend1: 12 requests (60%) ██████████████████████████████
  backend2:  4 requests (20%) ██████████
  backend3:  4 requests (20%) ██████████

💡 backend1 gets ~60% of traffic (weight 3/5).
   Use this when some servers are more powerful than others.


---
## Part 3: Health Checks

Load balancers don't just distribute traffic — they also **monitor backend health**. If a server crashes, the load balancer stops sending it traffic.

Nginx checks backends passively by default: if a backend returns an error or times out, nginx marks it as "down" temporarily.

Let's simulate this by stopping one backend:

In [9]:
import subprocess

# === Health Checks: Stop a backend and watch nginx adapt ===

def hit(n=9):
    """Send n requests through nginx and return how many each backend served."""
    counts = Counter()
    for _ in range(n):
        try:
            counts[requests.get(NGINX_URL, timeout=2).json()["server"]] += 1
        except Exception as e:
            counts[f"error:{type(e).__name__}"] += 1
    return dict(counts)

print("BEFORE stopping backend2:")
print(f"  Servers hit: {hit()}\n")

# Stop backend2 — nginx will detect the failure on the next request that lands there
print("Stopping backend2...")
subprocess.run(["docker", "stop", "net-backend2"], capture_output=True)
time.sleep(2)

print("\nAFTER stopping backend2 (nginx now routes around it):")
print(f"  Servers hit: {hit()}\n")

# Restart backend2 — Flask takes a moment to start listening on :5000
print("Restarting backend2...")
subprocess.run(["docker", "start", "net-backend2"], capture_output=True)

# Poll the backend directly until it answers, so we know it is healthy
print("   Waiting for backend2 to become healthy...")
for i in range(30):
    try:
        r = requests.get("http://localhost:5002/health", timeout=1)
        if r.status_code == 200:
            print(f"   backend2 healthy after {i+1}s")
            break
    except Exception:
        time.sleep(1)
else:
    print("   backend2 did not come back in time")

# Nginx may still mark it down briefly (default fail_timeout=10s), so we send
# enough requests to give it a chance to bring backend2 back into rotation.
print("\nAFTER restarting backend2:")
print(f"  Servers hit (30 requests): {hit(30)}")
print("\nNginx automatically routes around dead servers and brings them back")
print("once they recover. This is passive health checking.")


BEFORE stopping backend2:


  Servers hit: {'backend2': 3, 'backend3': 3, 'backend1': 3}

Stopping backend2...



AFTER stopping backend2 (nginx now routes around it):


  Servers hit: {'error:ReadTimeout': 3, 'backend3': 3, 'backend1': 3}

Restarting backend2...


   Waiting for backend2 to become healthy...


   backend2 healthy after 2s

AFTER restarting backend2:


  Servers hit (30 requests): {'backend2': 10, 'backend3': 10, 'backend1': 10}

Nginx automatically routes around dead servers and brings them back
once they recover. This is passive health checking.


---
## Part 4: L4 vs L7 Load Balancers

| Feature | Layer 4 (Transport) | Layer 7 (Application) |
|---------|--------------------|-----------------------|
| Inspects | IP addresses, ports | HTTP headers, URLs, cookies |
| Speed | Very fast | Slightly slower |
| Routing | Random / hash-based | Content-based (URL paths, headers) |
| Best for | WebSockets, raw TCP | REST APIs, HTTP services |
| Example | AWS NLB, HAProxy (TCP mode) | Nginx, AWS ALB, HAProxy (HTTP mode) |

Our nginx setup is an **L7 load balancer** — it reads the URL path to decide which upstream group to use:
- `/` → round-robin
- `/least-conn/` → least connections
- `/ip-hash/` → sticky sessions
- `/weighted/` → weighted distribution

An L4 load balancer couldn't do this — it would just forward raw TCP connections without knowing the URL.

In [10]:
# === Proof that nginx inspects HTTP content (L7) ===
# Look at the headers the backend receives — nginx adds its own.

r = requests.get(f"{NGINX_URL}/headers")
headers = r.json()["headers"]

print("Headers received by the backend:\n")
for key, value in headers.items():
    marker = " ← added by nginx" if key.startswith("X-") else ""
    print(f"  {key}: {value}{marker}")

print("\n💡 Nginx adds X-Real-IP, X-Forwarded-For, and X-Load-Balancer headers.")
print("   An L4 LB wouldn't be able to add these — it doesn't understand HTTP.")

Headers received by the backend:

  Accept: */*
  Accept-Encoding: gzip, deflate
  Host: localhost
  User-Agent: python-requests/2.33.1
  X-Forwarded-For: 192.168.65.1 ← added by nginx
  X-Load-Balancer: nginx-roundrobin ← added by nginx
  X-Real-Ip: 192.168.65.1 ← added by nginx

💡 Nginx adds X-Real-IP, X-Forwarded-For, and X-Load-Balancer headers.
   An L4 LB wouldn't be able to add these — it doesn't understand HTTP.


---
## Part 5: Forward Proxy vs Reverse Proxy

You have been using nginx as a **reverse proxy** for this whole lab. There is also a **forward proxy**, which is a different beast.

```
Forward Proxy  (sits in front of CLIENTS)        Reverse Proxy  (sits in front of SERVERS)

 Client A -+                                                          +-> backend1
 Client B -+--> Forward Proxy --> Internet      Reverse Proxy (nginx) +-> backend2
 Client C -+    (corporate firewall,                                  +-> backend3
                school content filter,            ^
                VPN/anonymizer)                   one public IP, many backends hidden
```

| | Forward Proxy | Reverse Proxy |
|---|---|---|
| Hides | The **client** from the server | The **server(s)** from the client |
| Configured by | The client (or its network) | The server operator |
| Examples | Squid, corporate web filter, VPN | nginx, HAProxy, AWS ALB, Cloudflare |
| Typical use | Privacy, content filtering, caching outbound traffic | Load balancing, TLS termination, caching inbound traffic |

### Why this matters in system design

- **Reverse proxies** are the entry point of nearly every production web service. They handle TLS, load balance, rate-limit, and cache.
- **CDNs** (Cloudflare, CloudFront, Fastly) are massive geographically distributed reverse proxies that also cache static content close to users.
- **API Gateways** (Kong, AWS API Gateway, Apigee) are reverse proxies with extra features like auth, request transformation, and quotas.


In [11]:
# === Reverse proxy vs direct: notice the IP the backend sees ===
# Through nginx (reverse proxy), the backend sees nginx's IP, NOT your client IP.
# That is one of the things that makes a reverse proxy 'hide' your backends.

via_proxy = requests.get(f"{NGINX_URL}/headers").json()
direct    = requests.get("http://localhost:5001/headers").json()

print("Via nginx (reverse proxy):")
print(f"  Host header:        {via_proxy['headers'].get('Host')}")
print(f"  X-Real-IP (you):    {via_proxy['headers'].get('X-Real-Ip')}")
print(f"  X-Forwarded-For:    {via_proxy['headers'].get('X-Forwarded-For')}")

print("\nDirectly to backend1 (bypassing nginx):")
print(f"  Host header:        {direct['headers'].get('Host')}")
print(f"  X-Real-IP:          {direct['headers'].get('X-Real-Ip', '(none -- only nginx adds this)')}")

print("\nWith a reverse proxy, backends only ever see the proxy's IP.")
print("The proxy preserves the original client IP in X-Forwarded-For so apps")
print("can still log/rate-limit by client. Trust this header carefully -- only")
print("when the request actually came from a proxy you control!")


Via nginx (reverse proxy):
  Host header:        localhost
  X-Real-IP (you):    192.168.65.1
  X-Forwarded-For:    192.168.65.1

Directly to backend1 (bypassing nginx):
  Host header:        localhost:5001
  X-Real-IP:          (none -- only nginx adds this)

With a reverse proxy, backends only ever see the proxy's IP.
The proxy preserves the original client IP in X-Forwarded-For so apps
can still log/rate-limit by client. Trust this header carefully -- only
when the request actually came from a proxy you control!


---
## 🎓 Key Takeaways

1. **DNS** translates names to IPs and acts as basic client-side load balancing
2. **Dedicated load balancers** (like nginx) give you more control over traffic distribution
3. **Round-robin** is the simplest and default algorithm — great for stateless services
4. **Least-connections** is better when request durations vary
5. **IP hash** provides sticky sessions for stateful backends
6. **Health checks** automatically route around failed servers
7. **L7 LBs** understand HTTP and can route by URL/headers; **L4 LBs** are faster but simpler

### Interview Tips
- Default to an **L7 load balancer** for HTTP/REST traffic
- Use **L4** for WebSocket or raw TCP connections
- Mention **DNS** for avoiding a single point of failure with your load balancers
- **Health checks** are essential — always mention them when discussing reliability